In [ ]:
# DFU Phase-5 Paper Evidence Packaging — one cell, no training/inference
import urllib.request, hashlib, base64, zlib

VERSION = "DFU_PHASE5_PAPER_EVIDENCE_LOADER_V1_20260812"
SOURCE_COMMIT = "1d763da5d8b0c212166a599d9e8a16de905a0e49"
PARTS = [
    ("scripts/phase5_paper_evidence_v1_payload/part_00.txt","b084ecedb84c69965417ba222fa728983659a1b3"),
    ("scripts/phase5_paper_evidence_v1_payload/part_01.txt","dc065293080176b2eee46157fcb1333cc695f747"),
    ("scripts/phase5_paper_evidence_v1_payload/part_02.txt","2d30de976b6698d5bcd63e935ab1466c8b2cbee3"),
    ("scripts/phase5_paper_evidence_v1_payload/part_03.txt","e51fb346a4be9cc2e7237b9fb98fcf90496ea150"),
]
EXPECTED_PAYLOAD_SHA256 = "4cd29975969b670db050a498833e81329d71ae899323099534cfd5edf41cd0ab"
EXPECTED_SOURCE_SHA256 = "7052d18cbbb3821d7b704fb68e1b94e863ec4ed4b0302214c311032231c077bd"
BASE=f"https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/{SOURCE_COMMIT}/"

def git_blob_sha(raw):
    return hashlib.sha1(b"blob "+str(len(raw)).encode()+b"\0"+raw).hexdigest()

print("="*100)
print(VERSION)
print("POST-HOC PACKAGING ONLY | NO TRAINING | NO CNN INFERENCE | DRIVE ARTIFACTS ONLY")
print("="*100)
chunks=[]
for path, expected in PARTS:
    raw=urllib.request.urlopen(BASE+path, timeout=120).read()
    actual=git_blob_sha(raw)
    if actual != expected:
        raise RuntimeError(f"Payload fragment mismatch: {path} expected={expected} actual={actual}")
    print("Payload fragment PASS:",path,actual)
    chunks.append(raw)
payload=b"".join(chunks)
psha=hashlib.sha256(payload).hexdigest()
if psha != EXPECTED_PAYLOAD_SHA256:
    raise RuntimeError(f"Payload SHA mismatch: expected={EXPECTED_PAYLOAD_SHA256} actual={psha}")
print("Combined payload SHA256: PASS",psha)
source=zlib.decompress(base64.b64decode(payload,validate=True))
ssha=hashlib.sha256(source).hexdigest()
if ssha != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(f"Decoded source SHA mismatch: expected={EXPECTED_SOURCE_SHA256} actual={ssha}")
print("Decoded source SHA256: PASS",ssha)
text=source.decode("utf-8")
compile(text,"dfu_phase5_paper_evidence_v1.py","exec")
print("Decoded source compile: PASS")
print("Starting Phase-5 packaging...")
exec(compile(text,"dfu_phase5_paper_evidence_v1.py","exec"),globals())
